In [1]:
pip install ultralytics roboflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from roboflow import Roboflow
rf = Roboflow(api_key="oIrHO5JXJh07U9INuXkw")
project = rf.workspace("puc-doiv").project("tcg-detector-manual")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Tcg-detector-manual-1 in yolov8:: 100%|██████████| 605/605 [00:00<00:00, 1948.69it/s]


In [8]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  
results = model.train(
    data="Tcg-detector-manual-1/data.yaml",
    epochs=150,
    batch=-1,
    imgsz=640,
    patience=30,      
    name="meu_treino"
)

New https://pypi.org/project/ultralytics/8.4.82 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.12.10 torch-2.12.1+cpu CPU (Intel Core Ultra 7 155H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Tcg-detector-manual-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m

In [9]:
metrics = model.val()  # avalia no conjunto de validação

results = model.predict(source="1test_img.png", save=True)

Ultralytics 8.4.75  Python-3.12.10 torch-2.12.1+cpu CPU (Intel Core Ultra 7 155H)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 392.962.0 MB/s, size: 39.3 KB)
val: Scanning D:\Documents\GitHub\tcg-card-detector\src\Tcg-detector-manual-1\valid\labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 8, len(boxes) = 372. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0s/it 3.9s6.8s
                   all         30        372      0.941        0.9      0.949      0.656
Speed: 0.8ms preprocess, 124.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\D

In [11]:
from ultralytics import YOLO

##model = YOLO("..\runs\detect\meu_treino-3\weights\best.pt")

conjuntos = ["train", "val", "test"]
resultados = {}

for split in conjuntos:
    m = model.val(data="Tcg-detector-manual-1/data.yaml", split=split, verbose=False)
    resultados[split] = {
        "mAP50":     round(m.box.map50, 4),
        "mAP50-95":  round(m.box.map, 4),
        "Precisão":  round(m.box.mp, 4),
        "Recall":    round(m.box.mr, 4),
    }

# Exibe tabela
print(f"{'Conjunto':<12} {'mAP50':<10} {'mAP50-95':<12} {'Precisão':<12} {'Recall'}")
print("-" * 55)
for split, metricas in resultados.items():
    print(f"{split:<12} {metricas['mAP50']:<10} {metricas['mAP50-95']:<12} {metricas['Precisão']:<12} {metricas['Recall']}")

Ultralytics 8.4.75  Python-3.12.10 torch-2.12.1+cpu CPU (Intel Core Ultra 7 155H)
val: Fast image access  (ping: 0.10.0 ms, read: 53.361.2 MB/s, size: 43.3 KB)
val: Scanning D:\Documents\GitHub\tcg-card-detector\src\Tcg-detector-manual-1\train\labels.cache... 240 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 240/240  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 249, len(boxes) = 3003. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.4s/it 36.6s2.5ss
                   all        240       3003      0.956      0.925      0.968      0.728
Speed: 0.7ms preprocess, 141.6ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to D:\Documents\GitHub\tcg-card-detector\runs\detect\val-4
Ultralytics 8.4.75  P